#  Análisis Táctico, Perfilamiento SOM y Scouting (Moneyball)

## 1. Importación de la Base de Datos Clusterizada

En este script nos desvinculamos del entrenamiento geométrico de la red neuronal. Nuestro único objetivo es leer la base de datos resultante del **Script 4** y traducir las agrupaciones matemáticas (Arquetipos SOM) al idioma real del fútbol.

El código está diseñado para ser **dinámico**: detectará automáticamente cuántos clústeres se generaron en el paso anterior, permitiéndonos cambiar la configuración en el futuro sin tener que reescribir este cuaderno.

In [6]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print(" 📥 CARGANDO ECOSISTEMA TÁCTICO")
print("="*80)

# 1. Definir la ruta del archivo exportado en el Script 4
ruta_csv = '/content/drive/MyDrive/dataset_final_clusters.csv'

# 2. Cargar el DataFrame
df_clusters = pd.read_csv(ruta_csv)

# 3. Detección Dinámica de Clústeres
total_clusters = df_clusters['Arquetipo_SOM'].nunique()
lista_clusters = sorted(df_clusters['Arquetipo_SOM'].unique())

print(f"✅ Dataset cargado exitosamente.")
print(f"✅ Total de jugadores en la base: {df_clusters.shape[0]}")
print(f"✅ Arquetipos Tácticos (Clústeres) detectados: {total_clusters}")
print(f"   IDs de los grupos: {lista_clusters}")
print("="*80)

# Mostramos una muestra rápida para confirmar que todo esté en orden
display(df_clusters[['player', 'team', 'position_detail', 'Arquetipo_SOM']].head())

 📥 CARGANDO ECOSISTEMA TÁCTICO
✅ Dataset cargado exitosamente.
✅ Total de jugadores en la base: 1699
✅ Arquetipos Tácticos (Clústeres) detectados: 7
   IDs de los grupos: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]


,player,team,position_detail,Arquetipo_SOM
0,Aaron Cresswell,West Ham United,Left Back,0
1,Aaron Lennon,Everton,Center Attacking Midfield,2
2,Aaron Ramsey,Arsenal,Center Attacking Midfield,2
3,Abdelaziz Barrada,Marseille,Left Center Midfield,6
4,Abdelaziz Barrada,Olympique de Marseille,Center Attacking Midfield,6


## 2. Radiografía de los Arquetipos (Z-Score y Percentiles)

Para entender qué rol táctico juega cada grupo en la cancha, agruparemos a los jugadores por su `Arquetipo_SOM` y calcularemos el promedio de sus métricas.

Dado que nuestros datos están estandarizados, utilizamos una doble lectura para máxima claridad:
1. **Z-Score (El Valor Crudo):** Indica a cuántas desviaciones estándar está el clúster respecto al jugador europeo promedio (0.00).
2. **Percentiles (La Traducción Visual):** Usando la Función de Distribución Acumulada Gaussiana (CDF), traducimos el Z-Score a un rango del 0 al 100. Un percentil 90 indica que ese grupo es superior al 90% de la liga en esa métrica.

Además, rankearemos los clústeres desde el que tiene mayor **Impacto Global** (los "todoterreno" o estrellas absolutas) hasta los de menor impacto (jugadores de rol muy específico o defensores rígidos).

In [7]:
print("="*85)
print(f" 🏆 REPORTE DESCRIPTIVO: RADIOGRAFÍA DE LOS {total_clusters} ARQUETIPOS TÁCTICOS")
print("="*85)

# 1. DEFINICIÓN DE LAS DIMENSIONES TÁCTICAS
dimensiones_tacticas = {
    '⚽ ATAQUE / FINALIZACIÓN': ['xG P90', 'xG por Tiro'],
    '🧠 CREACIÓN / DISTRIBUCIÓN': ['Asistencias a Tiro P90', 'Pases Último Tercio P90', 'Pases Progresivos P90', '% Pases Seguridad', '% Centralidad Equipo'],
    '🛡️ DEFENSA / DESTRUCCIÓN': ['Tackles Ganados PAdj P90', 'Intercepciones PAdj P90', 'Recup. Último Tercio P90'],
    '🏃 MOVILIDAD / RETENCIÓN': ['Conducciones Progresivas P90', 'Pérdidas de Balón P90', '% Pases Bajo Presión', 'Faltas Recibidas P90']
}
#
columnas_metricas = [col for dims in dimensiones_tacticas.values() for col in dims]

# 2. CÁLCULO DE RENDIMIENTO E IMPACTO
# Score base de cada jugador (promedio de todas sus estadísticas)
df_clusters['Score_Jugador'] = df_clusters[columnas_metricas].mean(axis=1)

# Agrupamos por clúster y promediamos sus variables
perfiles_descriptivos = df_clusters.groupby('Arquetipo_SOM')[columnas_metricas].mean()

# Calculamos el Impacto Global del clúster
perfiles_descriptivos['Impacto_Total_Cluster'] = perfiles_descriptivos.mean(axis=1)

# Ordenamos los clústeres del más influyente al menos influyente
clusters_ordenados = perfiles_descriptivos.sort_values(by='Impacto_Total_Cluster', ascending=False).index

# 3. BUCLE DINÁMICO DE IMPRESIÓN (Se adapta a la cantidad de clústeres que existan)
for ranking_pos, cluster_id in enumerate(clusters_ordenados, start=1):

    # Extraemos información vital del clúster actual
    jugadores_del_grupo = df_clusters[df_clusters['Arquetipo_SOM'] == cluster_id]
    cantidad_jugadores = len(jugadores_del_grupo)

    medias_cluster = perfiles_descriptivos.loc[cluster_id]
    score_impacto_cluster = medias_cluster['Impacto_Total_Cluster']
    percentil_global = norm.cdf(score_impacto_cluster) * 100

    print(f"\n==================== #{ranking_pos} | ARQUETIPO TÁCTICO {cluster_id} ({cantidad_jugadores} jugadores) ====================")
    print(f"🌟 IMPACTO GLOBAL DEL GRUPO: {score_impacto_cluster:+.2f} Z-Score (Percentil Promedio: {percentil_global:.0f})")

    # --- CÁLCULO DE SCORES MACRO-DIMENSIONALES ---
    scores_dimensiones = {}
    for dim_nombre, vars_en_dim in dimensiones_tacticas.items():
        score_promedio = medias_cluster[vars_en_dim].mean()
        scores_dimensiones[dim_nombre] = score_promedio

    print("\n📊 PERFIL DEL CLÚSTER (Fuerzas Tácticas):")
    resumen_macro = " | ".join([f"{dim.split(' ')[1]} {norm.cdf(score)*100:.0f}%" for dim, score in scores_dimensiones.items()])
    print(f"   {resumen_macro}")

    # --- DESGLOSE DE VARIABLES ---
    print("\n   🔍 Desglose de Variables (Promedio del Grupo):")
    for dim_nombre, vars_en_dim in dimensiones_tacticas.items():
        print(f"      {dim_nombre}:")
        valores_dim = medias_cluster[vars_en_dim].sort_values(ascending=False)
        for var_name, valor_z in valores_dim.items():
            indicador = "🟢" if valor_z > 0 else "🔴"
            percentil = norm.cdf(valor_z) * 100
            print(f"         {indicador} {var_name:<30} : {valor_z:+.2f} Z (Pc. {percentil:02.0f})")

    # --- JUGADORES ÉLITE DEL GRUPO ---
    print("\n   ⭐ Top 5 Jugadores (Máximo Rendimiento del Grupo):")
    top_5 = jugadores_del_grupo.sort_values(by='Score_Jugador', ascending=False).head(5)

    for index, row in top_5.iterrows():
        nombre = row['player']
        equipo = row['team']
        score_individual = row['Score_Jugador']
        percentil_ind = norm.cdf(score_individual) * 100
        print(f"      🏃 {nombre:<25} ({equipo:<15}) | Impacto Base: {score_individual:+.2f} Z (Pc. {percentil_ind:.0f})")
    print("-" * 85)

 🏆 REPORTE DESCRIPTIVO: RADIOGRAFÍA DE LOS 7 ARQUETIPOS TÁCTICOS

==================== #1 | ARQUETIPO TÁCTICO 0 (312 jugadores) ====================
🌟 IMPACTO GLOBAL DEL GRUPO: +1.10 Z-Score (Percentil Promedio: 86)

📊 PERFIL DEL CLÚSTER (Fuerzas Tácticas):
   ATAQUE 77% | CREACIÓN 86% | DEFENSA 93% | MOVILIDAD 85%

   🔍 Desglose de Variables (Promedio del Grupo):
      ⚽ ATAQUE / FINALIZACIÓN:
         🟢 xG P90                         : +0.92 Z (Pc. 82)
         🟢 xG por Tiro                    : +0.53 Z (Pc. 70)
      🧠 CREACIÓN / DISTRIBUCIÓN:
         🟢 Pases Último Tercio P90        : +1.39 Z (Pc. 92)
         🟢 Asistencias a Tiro P90         : +1.30 Z (Pc. 90)
         🟢 Pases Progresivos P90          : +1.30 Z (Pc. 90)
         🟢 % Centralidad Equipo           : +1.29 Z (Pc. 90)
         🟢 % Pases Seguridad              : +0.02 Z (Pc. 51)
      🛡️ DEFENSA / DESTRUCCIÓN:
         🟢 Intercepciones PAdj P90        : +1.59 Z (Pc. 94)
         🟢 Tackles Ganados PAdj P90       : +1.57